# Notebook 6 - Computer Vision

Roadmap role: Week 6. This Colab workflow preserves dataset provenance, official splits, model settings, raw outputs, and negative results. Agriculture-Vision is a semantic-segmentation benchmark; generic YOLO inference here is only the roadmap-required smoke-test baseline.

## 1. Clone The Week 6 Branch

Run this in a fresh Colab runtime. Raw imagery and generated outputs stay in the runtime and are not committed.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO = Path('/content/shepherd-ai')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', 'codex/week6-vision', 'https://github.com/cyberuniversal/shepherd-ai.git', str(REPO)], check=True)
os.chdir(REPO)
print('repository', Path.cwd())
print('commit', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


## 2. Review And Accept Agriculture-Vision Terms

Official terms: https://intelinair-data-releases.s3.amazonaws.com/agriculture-vision/cvpr_paper_2020/Agriculture-Vision%20Dataset%20Terms%20of%20Use.pdf

Downloading signifies agreement. The terms permit limited non-commercial research use and prohibit redistribution. Only continue if you personally reviewed and accept them.

In [ ]:
TERMS_ACKNOWLEDGMENT = input('Type I ACCEPT AGRICULTURE-VISION TERMS after reviewing them: ').strip()
if TERMS_ACKNOWLEDGMENT != 'I ACCEPT AGRICULTURE-VISION TERMS':
    raise RuntimeError('Terms were not accepted; dataset acquisition stopped.')
print('Terms acknowledged for this Colab session.')


## 3. Download And Extract The Official 2017 Archive

The archive is approximately 1.88 GB. The download comes directly from the official IntelinAir AWS bucket. Its SHA-256 is recorded after download.

In [ ]:
import hashlib
import tarfile
from urllib.request import urlretrieve

if TERMS_ACKNOWLEDGMENT != 'I ACCEPT AGRICULTURE-VISION TERMS':
    raise RuntimeError('Terms acknowledgment is required in this session.')
DATASET_DIR = REPO / 'datasets' / 'aerial_images' / 'agriculture-vision'
DATASET_DIR.mkdir(parents=True, exist_ok=True)
BASE = 'https://intelinair-data-releases.s3.amazonaws.com/agriculture-vision/cvpr_paper_2020/Dataset'
ARCHIVE = DATASET_DIR / 'data2017_miniscale.tar.gz'
SPLITS = DATASET_DIR / 'data2017_splits.json'
if not ARCHIVE.exists():
    urlretrieve(f'{BASE}/data2017_miniscale.tar.gz', ARCHIVE)
if not SPLITS.exists():
    urlretrieve(f'{BASE}/data2017_splits.json', SPLITS)
digest = hashlib.sha256()
with ARCHIVE.open('rb') as handle:
    for chunk in iter(lambda: handle.read(1024 * 1024), b''):
        digest.update(chunk)
print('archive_sha256', digest.hexdigest())
EXTRACTED = DATASET_DIR / 'data2017'
if not EXTRACTED.exists():
    EXTRACTED.mkdir()
    with tarfile.open(ARCHIVE, 'r:gz') as archive:
        archive.extractall(EXTRACTED, filter='data')
print('extracted_to', EXTRACTED)


## 4. Build And Validate A Leakage-Safe Subset Manifest

The official farmland-level split JSON is authoritative. Selection is deterministic and limited to 10 RGB images per split. Every selected image receives a SHA-256 digest.

In [ ]:
!python scripts/prepare_agriculture_vision_subset.py --dataset-dir datasets/aerial_images/agriculture-vision/data2017 --dataset-root . --split-json datasets/aerial_images/agriculture-vision/data2017_splits.json --output datasets/aerial_images/manifest.jsonl --max-per-split 10 --accept-terms
!python scripts/validate_vision_manifest.py --manifest datasets/aerial_images/manifest.jsonl --dataset-root . --summary-output outputs/evaluations/week6_vision_manifest_summary.json


## 5. Run The YOLO Smoke-Test Baseline

Detection counts and confidence values are pipeline outputs, not Agriculture-Vision anomaly performance. Zero detections are preserved as a valid negative result.

In [ ]:
!python -m pip install -q -e '.[vision]'
!python scripts/run_yolo_detection.py --manifest datasets/aerial_images/manifest.jsonl --dataset-root . --model yolov8n.pt --confidence 0.25 --device 0 --predictions-output outputs/evaluations/week6_yolo_detections.jsonl --summary-output outputs/evaluations/week6_yolo_detection_summary.json --annotated-dir outputs/visualizations/week6_yolo
